# DisasterScout — Model Training
This notebook covers the ML foundation for DisasterScout (Phase 0). It sets up the PyTorch U-Net model, loads the xBD dataset, and provides the training loop skeleton.

In [ ]:
!pip install segmentation-models-pytorch torchgeo rasterio albumentations geopandas

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

In [ ]:
# check for GPU
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(torch.cuda.get_device_name(0))
device = torch.device('cuda' if cuda_available else 'cpu')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# User should change this to their specific Drive path
XBD_PATH = '/content/drive/MyDrive/xBD'

In [ ]:
class xBDDataset(torch.utils.data.Dataset):
    """
    xBD Dataset class (custom implementation without TorchGeo).
    Note: Using concatenated 6-channel input, not Siamese architecture, for speed.
    """
    def __init__(self, root, split='train', disaster_types=None):
        self.root = root
        self.split = split
        # xBD data generally looks like:
        # xBD/<disaster_type>/images/<name>_pre_disaster.png
        # xBD/<disaster_type>/images/<name>_post_disaster.png
        # xBD/<disaster_type>/labels/<name>_post_disaster.json (damage labels)

    def __len__(self):
        return 0

    def __getitem__(self, idx):
        # Return dict format:
        # {"image": tensor(6, 512, 512), "mask": tensor(512, 512), "name": str}
        pass

In [ ]:
def show_sample(pre_img, post_img, mask):
    # Visualize a sample pre/post and its mask side-by-side
    # using damage colors: no_change=gray, flood=blue, structural=red, road=orange
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(pre_img)
    axes[0].set_title('Pre Disaster')
    axes[1].imshow(post_img)
    axes[1].set_title('Post Disaster')
    axes[2].imshow(mask)
    axes[2].set_title('Damage Mask')
    plt.show()

In [ ]:
# Build Model
model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights='imagenet',
    in_channels=6,
    classes=4,
    activation=None
)
model = model.to(device)
print("Model summary:\n", model)

In [ ]:
from torch.utils.data import DataLoader

train_dataset = xBDDataset(XBD_PATH, split='train', 
  disaster_types=['turkey_earthquake'])
val_dataset = xBDDataset(XBD_PATH, split='tier3',
  disaster_types=['turkey_earthquake'])

train_loader = DataLoader(train_dataset, batch_size=4, 
  shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, 
  shuffle=False, num_workers=2)

def compute_iou(pred_mask, true_mask, num_classes=4):
  ious = []
  pred = torch.argmax(pred_mask, dim=1)
  for cls in range(1, num_classes):  # skip class 0 (no change)
    pred_cls = (pred == cls)
    true_cls = (true_mask == cls)
    intersection = (pred_cls & true_cls).sum().float()
    union = (pred_cls | true_cls).sum().float()
    if union == 0:
      continue
    ious.append((intersection / union).item())
  return sum(ious) / len(ious) if ious else 0.0

NUM_EPOCHS = 25

class_weights = torch.tensor([0.1, 1.0, 1.5, 1.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
  optimizer, mode='min', patience=3, factor=0.5)

best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
  # --- Training phase ---
  model.train()
  train_loss = 0.0
  for batch in train_loader:
    images = batch['image'].to(device)   # (B, 6, 512, 512)
    masks = batch['mask'].to(device)     # (B, 512, 512) long tensor
    
    optimizer.zero_grad()
    outputs = model(images)              # (B, 4, 512, 512)
    loss = criterion(outputs, masks)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  
  avg_train_loss = train_loss / len(train_loader)
  
  # --- Validation phase ---
  model.eval()
  val_loss = 0.0
  val_iou = 0.0
  with torch.no_grad():
    for batch in val_loader:
      images = batch['image'].to(device)
      masks = batch['mask'].to(device)
      outputs = model(images)
      loss = criterion(outputs, masks)
      val_loss += loss.item()
      val_iou += compute_iou(outputs, masks)
  
  avg_val_loss = val_loss / len(val_loader)
  avg_val_iou = val_iou / len(val_loader)
  scheduler.step(avg_val_loss)
  
  # Save best model
  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    torch.save(model.state_dict(), 
      '/content/drive/MyDrive/disasterscout_best.pth')
    print(f"  ✓ Best model saved (val_loss: {avg_val_loss:.4f})")
  
  print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val IoU: {avg_val_iou:.4f}")

print("Training complete. Best checkpoint saved to Google Drive.")


In [ ]:
print("WARNING: IoU below 0.35. Try: reduce lr to 1e-5, add more augmentation, check dataset path is correct.")


In [ ]:
# Save checkpoint
torch.save(model.state_dict(), '/content/drive/MyDrive/disasterscout_checkpoint.pth')
print("Saved to Drive")